In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction — Logistic Regression Baseline

Logistic Regression with domain-informed missing value handling, VIF-based feature selection,
and standard scaling. No hyperparameter tuning beyond default regularization.
Purpose: linear baseline for comparison with tree-based models.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"
STUDY_NAME = "lr_baseline_metabolic_v2"
RANDOM_STATE = 37

USE_CALIBRATION = False

# Columns to drop before modeling
DROP_COLUMNS = [
    "has_diabetes_or_prediabetes",  # target
    "cycle",                        # metadata
    "survey_weight",                # not a feature
    "lab_positive",                 # lab target, NOT a feature (requires blood draw)
    "undiagnosed",                  # derived target, NOT a feature
    "waist_to_height_ratio",        # feature engineering — not for baseline
    "age_bmi_interaction",          # feature engineering — not for baseline
    "age_young_adult",              # age bins — not for baseline
    "age_middle_age",               # age bins — not for baseline
    "age_senior",                   # age bins — not for baseline
    "age_elderly",                  # age bins — not for baseline
    "binge_episodes_month",         # 47% missing, ambiguous skip logic
]

# Physical activity columns where NaN = "doesn't do this activity"
PA_ZERO_FILL_COLUMNS = [
    "vigorous_minutes_per_week",
    "moderate_minutes_per_week",
]

# Alcohol column where NaN = "doesn't drink"
ALCOHOL_ZERO_FILL_COLUMNS = [
    "drinks_per_day",
]

# VIF threshold
VIF_THRESHOLD = 5

In [ ]:
data = pd.read_csv("../../../dataset/processed_data_combined_metabolic_history.csv")
print(f"Raw data shape: {data.shape}")
data.head()

In [ ]:
# Physical activity: NaN means "doesn't do this activity" → fill with 0
for col in PA_ZERO_FILL_COLUMNS:
    if col in data.columns:
        n_missing = data[col].isna().sum()
        data[col] = data[col].fillna(0)
        print(f"{col}: filled {n_missing} NaNs with 0 (no activity)")

# Alcohol: NaN means "doesn't drink" → fill with 0
for col in ALCOHOL_ZERO_FILL_COLUMNS:
    if col in data.columns:
        n_missing = data[col].isna().sum()
        data[col] = data[col].fillna(0)
        print(f"{col}: filled {n_missing} NaNs with 0 (non-drinker)")

# Create binary indicators for physical activity
data["does_vigorous"] = (data["vigorous_minutes_per_week"] > 0).astype(int)
data["does_moderate"] = (data["moderate_minutes_per_week"] > 0).astype(int)

print(f"\nAdded does_vigorous: {data['does_vigorous'].mean():.1%} active")
print(f"Added does_moderate: {data['does_moderate'].mean():.1%} active")

In [ ]:
y = data["has_diabetes_or_prediabetes"]
X = data.drop(columns=DROP_COLUMNS, errors="ignore")

print(f"X shape: {X.shape}")
print(f"Prevalence: {y.mean():.1%}")
print(f"\nRemaining columns:\n{list(X.columns)}")

In [ ]:
# First split: separate out the untouchable test set (10%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)

# Second split: separate threshold-tuning validation
X_train_full, X_val_thresh, y_train_full, y_val_thresh = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.12,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

# Third split: separate early-stopping validation from training
X_train, X_val_es, y_train, y_val_es = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.125,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

print(f"Train: {len(X_train)} | Val ES: {len(X_val_es)} | Val Thresh: {len(X_val_thresh)} | Test: {len(X_test)}")
print(f"Prevalence — Train: {y_train.mean():.1%} | Val ES: {y_val_es.mean():.1%} | Val Thresh: {y_val_thresh.mean():.1%} | Test: {y_test.mean():.1%}")

# LR doesn't need early stopping -> recombine for more training data
X_train_lr = pd.concat([X_train, X_val_es])
y_train_lr = pd.concat([y_train, y_val_es])
print(f"LR effective training set: {len(X_train_lr)} (train + val_es recombined)")

In [ ]:
# Identify column types
cat_cols = X_train_lr.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train_lr.columns if c not in cat_cols]

# Fit imputers on recombined training data
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_lr_imp = X_train_lr.copy()
X_val_thresh_imp = X_val_thresh.copy()
X_test_imp = X_test.copy()

if num_cols:
    X_train_lr_imp[num_cols] = num_imputer.fit_transform(X_train_lr[num_cols])
    X_val_thresh_imp[num_cols] = num_imputer.transform(X_val_thresh[num_cols])
    X_test_imp[num_cols] = num_imputer.transform(X_test[num_cols])

if cat_cols:
    X_train_lr_imp[cat_cols] = cat_imputer.fit_transform(X_train_lr[cat_cols])
    X_val_thresh_imp[cat_cols] = cat_imputer.transform(X_val_thresh[cat_cols])
    X_test_imp[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# Verify no NaNs remain
assert X_train_lr_imp.isna().sum().sum() == 0, "Train still has NaNs!"
assert X_val_thresh_imp.isna().sum().sum() == 0, "Val thresh still has NaNs!"
assert X_test_imp.isna().sum().sum() == 0, "Test still has NaNs!"

print(f"After imputation — remaining NaN count: {X_train_lr_imp.isna().sum().sum()}")
print(f"Imputed columns (numeric, median): {num_cols}")

In [ ]:
# Scale first — VIF is sensitive to scale
scaler_vif = StandardScaler()
X_train_lr_scaled_vif = pd.DataFrame(
    scaler_vif.fit_transform(X_train_lr_imp[num_cols]),
    columns=num_cols,
    index=X_train_lr_imp.index,
)


def compute_vif(df):
    """Compute VIF for all columns in a DataFrame."""
    vif_data = pd.DataFrame()
    vif_data["Feature"] = df.columns
    vif_data["VIF"] = [
        variance_inflation_factor(df.values, i) for i in range(df.shape[1])
    ]
    return vif_data.sort_values("VIF", ascending=False)


# Iteratively drop highest VIF until all are below threshold
features_to_check = num_cols.copy()
dropped_features = []

while True:
    vif_df = compute_vif(X_train_lr_scaled_vif[features_to_check])
    max_vif = vif_df["VIF"].max()

    if max_vif <= VIF_THRESHOLD:
        break

    worst_feature = vif_df.iloc[0]["Feature"]
    print(f"Dropping {worst_feature} (VIF={max_vif:.1f})")
    features_to_check.remove(worst_feature)
    dropped_features.append(worst_feature)

print(f"\n--- VIF Summary ---")
print(f"Dropped {len(dropped_features)} features: {dropped_features}")
print(f"Remaining {len(features_to_check)} numeric features: {features_to_check}")
print(f"\nFinal VIF table:")
print(compute_vif(X_train_lr_scaled_vif[features_to_check]).to_string(index=False))

In [ ]:
# Keep surviving numeric features + any categorical features
final_features = features_to_check + cat_cols
print(f"Final feature set ({len(final_features)} features): {final_features}")

X_train_lr_final = X_train_lr_imp[final_features]
X_val_thresh_final = X_val_thresh_imp[final_features]
X_test_final = X_test_imp[final_features]

In [ ]:
scaler = StandardScaler()
X_train_lr_scaled = scaler.fit_transform(X_train_lr_final)
X_val_thresh_scaled = scaler.transform(X_val_thresh_final)
X_test_scaled = scaler.transform(X_test_final)

model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight="balanced",
)
model.fit(X_train_lr_scaled, y_train_lr)

print(f"Converged: {model.n_iter_[0]} iterations")
print(f"Coefficients shape: {model.coef_.shape}")

In [ ]:
coef_df = pd.DataFrame({
    "Feature": final_features,
    "Coefficient": model.coef_[0],
    "Abs_Coefficient": np.abs(model.coef_[0]),
}).sort_values("Abs_Coefficient", ascending=True)

fig_coef, ax = plt.subplots(figsize=(10, 8))
colors = ["red" if c < 0 else "steelblue" for c in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors)
ax.set_title("Logistic Regression Coefficients")
ax.set_xlabel("Coefficient (blue=increases risk, red=decreases risk)")
ax.axvline(x=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test_scaled)
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

recall_pos = recall_score(y_test, y_pred, pos_label=1)
recall_neg = recall_score(y_test, y_pred, pos_label=0)
print(f"Recall (positive): {recall_pos:.2%}")
print(f"Recall (negative): {recall_neg:.2%}")
print(f"y_test distribution:\n{y_test.value_counts()}")
print(f"y_pred distribution:\n{pd.Series(y_pred).value_counts()}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig_cm_05, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]).plot(cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — Threshold = 0.50\nRecall: {recall_pos:.2%}")
plt.tight_layout()
plt.show()

In [ ]:
# Threshold optimization
y_proba = model.predict_proba(X_val_thresh_scaled)[:, 1]

# Calculate PR curve
precisions, recalls, thresholds = precision_recall_curve(y_val_thresh, y_proba)
pr_auc = auc(recalls, precisions)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Precision-Recall Curve ---
ax1 = axes[0]
ax1.plot(recalls, precisions, "b-", linewidth=2, label=f"PR Curve (AUC={pr_auc:.3f})")
ax1.fill_between(recalls, precisions, alpha=0.2)

# Mark key threshold points
target_recalls = [0.80, 0.75, 0.70, 0.50]
colors = ["red", "orange", "green", "purple"]

for target, color in zip(target_recalls, colors):
    idx = np.where(recalls[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds[i]
        ax1.scatter(
            recalls[i],
            precisions[i],
            c=color,
            s=100,
            zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precisions[i]:.1%})",
        )

# Baseline (random classifier)
baseline = y_val_thresh.mean()
ax1.axhline(
    y=baseline,
    color="gray",
    linestyle="--",
    label=f"Baseline (prevalence={baseline:.1%})",
)

ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
ax1.set_title("Precision-Recall Curve (LR)", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# --- Plot 2: Threshold vs Metrics ---
ax2 = axes[1]

thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh = []
precision_at_thresh = []
f1_at_thresh = []
flagged_pct = []

for t in thresh_range:
    y_pred_t = (y_proba >= t).astype(int)
    r = recall_score(y_val_thresh, y_pred_t, zero_division=0)
    p = precision_score(y_val_thresh, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred_t.mean())

ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")

# Mark default 0.5 threshold
ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")

# Mark optimal threshold for 80% recall
target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
ax2.axvline(
    x=optimal_thresh,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)

ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title("Metrics vs Decision Threshold (LR)", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Print Summary Table ---
print("\n" + "=" * 70)
print("THRESHOLD ANALYSIS SUMMARY (LR)")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<12}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_proba >= thresh).astype(int)
    r = recall_score(y_val_thresh, y_pred_t, zero_division=0)
    p = precision_score(y_val_thresh, y_pred_t, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    n_flagged = y_pred_t.sum()
    pct_flagged = y_pred_t.mean() * 100
    print(
        f"{thresh:<12.2f} {r:<12.1%} {p:<12.1%} {f1:<12.3f} {n_flagged} ({pct_flagged:.1f}%)"
    )

print("=" * 70)
print(
    f"\nValidation set: {len(y_val_thresh)} samples, {y_val_thresh.sum()} diabetes cases ({y_val_thresh.mean():.1%} prevalence)"
)

In [ ]:
y_pred_final = (y_test_proba >= optimal_thresh).astype(int)

def bootstrap_metric(y_true, y_pred_or_proba, metric_fn, n_boot=2000, ci=0.95, random_state=RANDOM_STATE):
    """Compute metric with bootstrap confidence interval."""
    rng = np.random.RandomState(random_state)
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        try:
            scores.append(metric_fn(
                y_true.iloc[idx] if hasattr(y_true, "iloc") else y_true[idx],
                y_pred_or_proba[idx]
            ))
        except (ValueError, ZeroDivisionError):
            continue
    scores = np.array(scores)
    alpha = (1 - ci) / 2
    lo, hi = np.percentile(scores, [alpha * 100, (1 - alpha) * 100])
    return np.mean(scores), lo, hi

r_mean, r_lo, r_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: recall_score(yt, yp, zero_division=0))
p_mean, p_lo, p_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: precision_score(yt, yp, zero_division=0))
f1_mean, f1_lo, f1_hi = bootstrap_metric(y_test, y_pred_final, lambda yt, yp: f1_score(yt, yp, zero_division=0))
ap_mean, ap_lo, ap_hi = bootstrap_metric(y_test, y_test_proba, average_precision_score)
brier = brier_score_loss(y_test, y_test_proba)

print(f"\nFinal chosen threshold: {optimal_thresh:.4f}")
print(f"\n{'Metric':<20} {'Point Est.':<14} {'95% CI':<20}")
print("-" * 54)
print(f"{'Recall':<20} {r_mean:<14.2%} [{r_lo:.2%}, {r_hi:.2%}]")
print(f"{'Precision':<20} {p_mean:<14.2%} [{p_lo:.2%}, {p_hi:.2%}]")
print(f"{'F1 Score':<20} {f1_mean:<14.3f} [{f1_lo:.3f}, {f1_hi:.3f}]")
print(f"{'Average Precision':<20} {ap_mean:<14.3f} [{ap_lo:.3f}, {ap_hi:.3f}]")
print(f"{'Brier Score':<20} {brier:<14.4f}")

## Calibration Diagnostics

In [ ]:
from sklearn.calibration import calibration_curve

y_test_proba_for_cal = model.predict_proba(X_test_scaled)[:, 1]

fig_calibration, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Reliability Diagram ---
ax1 = axes[0]
fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_test_proba_for_cal, n_bins=10, strategy="uniform"
)
ax1.plot(mean_predicted_value, fraction_of_positives, "s-", label="Model", linewidth=2)
ax1.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
ax1.set_xlabel("Mean Predicted Probability", fontsize=12)
ax1.set_ylabel("Fraction of Positives", fontsize=12)
ax1.set_title("Reliability Diagram", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Plot 2: Predicted probability distribution ---
ax2 = axes[1]
ax2.hist(y_test_proba_for_cal[y_test == 0], bins=50, alpha=0.6, label="Negative", density=True)
ax2.hist(y_test_proba_for_cal[y_test == 1], bins=50, alpha=0.6, label="Positive", density=True)
ax2.axvline(x=optimal_thresh, color="red", linestyle="--", label=f"Threshold={optimal_thresh:.3f}")
ax2.set_xlabel("Predicted Probability", fontsize=12)
ax2.set_ylabel("Density", fontsize=12)
ax2.set_title("Predicted Probability Distribution by Class", fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_calibration_diag = fig_calibration
plt.show()

brier_test = brier_score_loss(y_test, y_test_proba_for_cal)
print(f"\nBrier Score (test set): {brier_test:.4f}")
print(f"  -> 0.0 = perfect calibration, {y_test.mean() * (1 - y_test.mean()):.4f} = baseline (prevalence-based)")

## SHAP Analysis

In [ ]:
import shap

# SHAP for Logistic Regression — LinearExplainer
X_train_lr_scaled_df = pd.DataFrame(
    X_train_lr_scaled,
    columns=final_features,
    index=X_train_lr_final.index,
)
X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=final_features,
    index=X_test_final.index,
)

masker = shap.maskers.Independent(X_train_lr_scaled_df, max_samples=100)
explainer = shap.LinearExplainer(model, masker)
shap_values_pos = explainer.shap_values(X_test_scaled_df)

fig_shap_global, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test_scaled_df, plot_type="bar", show=False, max_display=20)
plt.title("Global Feature Importance (mean |SHAP|)")
plt.tight_layout()
fig_shap_global = plt.gcf()
plt.show()

In [ ]:
# --- SHAP beeswarm plot ---
fig_shap_beeswarm, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values_pos, X_test_scaled_df, show=False, max_display=20)
plt.title("SHAP Beeswarm — Feature Impact on Positive Class Prediction")
plt.tight_layout()
fig_shap_beeswarm = plt.gcf()
plt.show()

# --- Dependence plots for top 3 features ---
mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
top_features = pd.Series(mean_abs_shap, index=X_test_scaled_df.columns).sort_values(ascending=False).head(3).index.tolist()

fig_shap_dep, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top_features):
    shap.dependence_plot(feat, shap_values_pos, X_test_scaled_df, ax=axes[i], show=False)
plt.suptitle("SHAP Dependence Plots — Top 3 Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

shap_importance = pd.Series(mean_abs_shap, index=X_test_scaled_df.columns).sort_values(ascending=False)
print("\nSHAP Global Importance (top 10):")
for feat, val in shap_importance.head(10).items():
    print(f"  {feat}: {val:.4f}")



## Model & Test Set Persistence

In [ ]:
import os
import joblib
from datetime import datetime

ARTIFACTS_DIR = "./artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_prefix = f"{STUDY_NAME}_{timestamp}"

model_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_model.joblib"
scaler_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_scaler.joblib"
test_bundle_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_test_bundle.joblib"
metadata_path = f"{ARTIFACTS_DIR}/{artifact_prefix}_metadata.joblib"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

test_bundle = {
    "X_test": X_test_scaled_df,
    "y_test": y_test,
    "y_test_proba": y_test_proba,
    "y_pred_final": y_pred_final,
    "feature_names": list(X_test_scaled_df.columns),
}
joblib.dump(test_bundle, test_bundle_path)

metadata = {
    "study_name": STUDY_NAME,
    "optimal_threshold": optimal_thresh,
    "use_calibration": USE_CALIBRATION,
    "random_state": RANDOM_STATE,
    "timestamp": timestamp,
    "final_features": final_features,
    "vif_threshold": VIF_THRESHOLD,
    "n_features_post_vif": len(final_features),
    "test_metrics": {
        "recall": r_mean,
        "recall_ci": (r_lo, r_hi),
        "precision": p_mean,
        "precision_ci": (p_lo, p_hi),
        "f1": f1_mean,
        "f1_ci": (f1_lo, f1_hi),
        "ap": ap_mean,
        "ap_ci": (ap_lo, ap_hi),
        "brier": brier,
    },
}
joblib.dump(metadata, metadata_path)

print("=" * 70)
print("ARTIFACTS SAVED")
print("=" * 70)
print(f"Model:        {model_path}")
print(f"Scaler:       {scaler_path}")
print(f"Test bundle:  {test_bundle_path}")
print(f"Metadata:     {metadata_path}")
print(f"\nTest set shape: {X_test_scaled_df.shape}")
print(f"Positive cases in test: {y_test.sum()} ({y_test.mean():.1%})")
print("=" * 70)

In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)
fig_cm_opt, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=["No Diabetes", "Diabetes"]).plot(cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix — Threshold = {optimal_thresh:.3f}\n(recall={r_mean:.2%}, precision={p_mean:.2%})")
plt.tight_layout()
plt.show()

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=ENTITY,
    name=f"{STUDY_NAME}-evaluation",
    job_type="evaluation",
    config={
        "model_type": "logistic_regression",
        "tuned": False,
        "class_weight": "balanced",
        "max_iter": 1000,
        "chosen_threshold": optimal_thresh,
        "use_calibration": USE_CALIBRATION,
        "vif_threshold": VIF_THRESHOLD,
        "dropped_features_vif": dropped_features,
        "dropped_columns_initial": DROP_COLUMNS,
        "pa_zero_fill": PA_ZERO_FILL_COLUMNS,
        "alcohol_zero_fill": ALCOHOL_ZERO_FILL_COLUMNS,
        "final_features": final_features,
        "n_features": len(final_features),
        "train_size": len(X_train),
        "val_es_size": len(X_val_es),
        "val_thresh_size": len(X_val_thresh),
        "test_size": len(X_test),
    },
)

In [ ]:
y_probas_both = model.predict_proba(X_test_scaled)
wandb.log({
    "pr_curve": wandb.plot.pr_curve(
        y_true=y_test.values,
        y_probas=y_probas_both,
        labels=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
wandb.log({
    "feature_importance_shap": wandb.Table(
        columns=["Feature", "Mean_Abs_SHAP"],
        data=[[feat, float(imp)] for feat, imp in shap_importance.items()],
    )
})

In [ ]:
wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        y_true=y_test.values,
        preds=y_pred_final,
        class_names=["No Diabetes", "Diabetes"],
    )
})

In [ ]:
y_val_proba = model.predict_proba(X_val_thresh_scaled)[:, 1]
threshold_data = []
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred_t = (y_val_proba >= t).astype(int)
    threshold_data.append([
        t,
        recall_score(y_val_thresh, y_pred_t, zero_division=0),
        precision_score(y_val_thresh, y_pred_t, zero_division=0),
        f1_score(y_val_thresh, y_pred_t, zero_division=0),
        y_pred_t.mean(),
    ])

wandb.log({
    "threshold_analysis": wandb.Table(
        columns=["Threshold", "Recall", "Precision", "F1", "Pct_Flagged"],
        data=threshold_data,
    )
})

In [ ]:
import os

chart_dir = "wandb_charts"
os.makedirs(chart_dir, exist_ok=True)

charts = {
    "confusion_matrix_050": fig_cm_05,
    "confusion_matrix_optimal": fig_cm_opt,
    "pr_threshold_analysis": fig,
    "lr_coefficients": fig_coef,
    "calibration_diagnostics": fig_calibration_diag,
    "shap_global_importance": fig_shap_global,
    "shap_beeswarm": fig_shap_beeswarm,
}

for name, figure in charts.items():
    figure.savefig(f"{chart_dir}/{name}.png", dpi=150, bbox_inches="tight")

artifact = wandb.Artifact(
    name=f"{STUDY_NAME}-charts",
    type="evaluation-charts",
    description="All evaluation charts for LR baseline model",
    metadata={
        "threshold_default": 0.5,
        "threshold_optimal": optimal_thresh,
        "test_recall": r_mean,
        "test_precision": p_mean,
        "dropped_features_vif": dropped_features,
    },
)
artifact.add_dir(chart_dir)
wandb.log_artifact(artifact)

wandb.log({name: wandb.Image(figure) for name, figure in charts.items()})

In [ ]:
y_pred_05 = (y_test_proba >= 0.5).astype(int)

wandb.summary.update({
    "test_recall": r_mean,
    "test_recall_ci_lo": r_lo,
    "test_recall_ci_hi": r_hi,
    "test_precision": p_mean,
    "test_precision_ci_lo": p_lo,
    "test_precision_ci_hi": p_hi,
    "test_f1": f1_mean,
    "test_f1_ci_lo": f1_lo,
    "test_f1_ci_hi": f1_hi,
    "test_ap": ap_mean,
    "test_ap_ci_lo": ap_lo,
    "test_ap_ci_hi": ap_hi,
    "test_brier": brier,
    "test_recall_at_050": recall_score(y_test, y_pred_05),
    "test_precision_at_050": precision_score(y_test, y_pred_05),
    "chosen_threshold": optimal_thresh,
    "use_calibration": USE_CALIBRATION,
    "test_size": len(y_test),
    "val_thresh_size": len(y_val_thresh),
    "train_size": len(X_train),
    "test_prevalence": float(y_test.mean()),
    "n_features": len(final_features),
    "n_features_dropped_vif": len(dropped_features),
    "model_type": "logistic_regression",
    "tuned": False,
})

wandb.finish()